# 04 — Baseline CNN Evaluation

Load the best checkpoint produced by notebook 02, run inference on the **test set**, and generate all diagnostic plots.

**Prerequisite:** the model package is installed in the active kernel and `artifacts/checkpoints/baseline_cnn_checkpoint.keras` exists (run notebook 02 first).

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, f1_score

from model_service.config import ModelServiceConfig
from model_service.preprocess.dataset_builder import build_pcam_datasets
from model_service.evaluation.plots import (
    plot_confusion_matrix,
    plot_roc,
    plot_pr_curve,
)

In [2]:
cfg = ModelServiceConfig()

checkpoint_path = cfg.paths.artifacts_checkpoints / "baseline_cnn_checkpoint.keras"
figures_dir = cfg.paths.artifacts_figures
figures_dir.mkdir(parents=True, exist_ok=True)

print("Loading checkpoint:", checkpoint_path)
model = tf.keras.models.load_model(checkpoint_path)
model.summary()

Loading checkpoint: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/checkpoints/baseline_cnn_checkpoint.keras


2026-04-23 00:54:32.646031: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-04-23 00:54:32.646053: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-04-23 00:54:32.646056: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2026-04-23 00:54:32.646244: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-23 00:54:32.646260: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,517 (252.02 KB)

 Trainable params: 21,505 (84.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 43,012 (168.02 KB)

In [3]:
_, _, test_ds, _ = build_pcam_datasets(download=False, use_efficientnet_preprocess=False)

In [4]:
# Collect ground-truth labels and predicted scores from the test set
y_true_list, y_score_list = [], []

for x_batch, y_batch in test_ds:
    preds = model.predict_on_batch(x_batch)   # shape (B, 1)
    y_score_list.append(preds.ravel())
    y_true_list.append(y_batch.numpy().ravel())

y_true  = np.concatenate(y_true_list).astype(np.int32)
y_score = np.concatenate(y_score_list).astype(np.float32)

print(f"Test samples: {len(y_true)}  |  Positive rate: {y_true.mean():.3f}")

2026-04-23 00:54:33.680377: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Test samples: 32768  |  Positive rate: 0.500


2026-04-23 00:54:39.330630: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [5]:
# Find the threshold that maximises F1 on the test set
thresholds = np.linspace(0.01, 0.99, 199)
f1_scores  = [f1_score(y_true, (y_score >= t).astype(int), zero_division=0) for t in thresholds]
best_threshold = float(thresholds[np.argmax(f1_scores)])
best_f1        = float(np.max(f1_scores))
print(f"Best F1 threshold: {best_threshold:.2f}  →  F1 = {best_f1:.4f}")

Best F1 threshold: 0.33  →  F1 = 0.8148


In [6]:
# ROC curve
plot_roc(
    y_true,
    y_score,
    out_path=figures_dir / "baseline_roc.png",
)
print("Saved:", figures_dir / "baseline_roc.png")

Saved: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/figures/baseline_roc.png


In [7]:
# Precision-Recall curve with best-F1 operating point
plot_pr_curve(
    y_true,
    y_score,
    best_f1_threshold=best_threshold,
    out_path=figures_dir / "baseline_pr_curve.png",
)
print("Saved:", figures_dir / "baseline_pr_curve.png")

Saved: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/figures/baseline_pr_curve.png


In [8]:
# Confusion matrix at the best-F1 threshold
y_pred = (y_score >= best_threshold).astype(np.int32)
cm     = confusion_matrix(y_true, y_pred)

plot_confusion_matrix(
    cm,
    labels=("Normal", "Metastatic"),
    out_path=figures_dir / "baseline_confusion_matrix.png",
)
print("Saved:", figures_dir / "baseline_confusion_matrix.png")
print(cm)

Saved: /Users/sandinosaso/code/sandinosaso/pathsight/artifacts/figures/baseline_confusion_matrix.png
[[12145  4246]
 [ 2198 14179]]
